# Simulation 3 — Ring resonator Q-factor and FSR

A **ring resonator** filters wavelengths: on resonance the round-trip phase satisfies $2\\pi r n_{\\mathrm{eff}} = m\\lambda$. The transmission spectrum shows periodic notches with free spectral range $\\mathrm{FSR} \\approx \\lambda^2 / (n_g L)$ and linewidth set by coupling and loss (Q-factor).

We inject a broadband Gaussian pulse, record bus flux, and fit a Lorentzian to the deepest resonance.

In [ ]:
import matplotlib.pyplot as plt
import tidy3d.web as web

from fdtd_pic.analytics.ring import analytical_fsr_hz, fit_resonance, measured_fsr
from fdtd_pic.config import RING_RADIUS
from fdtd_pic.plotting import apply_style, save_figure
from fdtd_pic.ring import build_ring_simulation, extract_transmission_spectrum

apply_style()

In [ ]:
sim = build_ring_simulation()
sim.plot(z=0)
plt.title('Ring resonator geometry')
plt.show()

In [ ]:
sim_data = web.run(sim, task_name='ring_broadband', verbose=True)
freqs, transmission = extract_transmission_spectrum(sim_data)

fig, ax = plt.subplots()
ax.plot(freqs / 1e12, transmission)
ax.set_xlabel('Frequency (THz)')
ax.set_ylabel('Through transmission')
ax.set_title('Ring transmission spectrum')
save_figure(fig, '../assets/ring_transmission_spectrum.png')
plt.show()

In [ ]:
fit = fit_resonance(freqs, transmission)
fsr_sim = measured_fsr(freqs, transmission)
fsr_analytical_hz = analytical_fsr_hz(RING_RADIUS)

print(f'Resonance f0 = {fit.f0/1e12:.3f} THz')
print(f'Q = {fit.q_factor:.0f}')
print(f'FSR (sim) ~ {fsr_sim/1e12:.4f} THz' if fsr_sim == fsr_sim else 'FSR (sim): n/a')
print(f'FSR (analytical) ~ {fsr_analytical_hz/1e12:.4f} THz')

fig, ax = plt.subplots()
ax.plot(freqs / 1e12, transmission, label='FDTD')
ax.axvline(fit.f0 / 1e12, color='r', ls='--', label=f'fit f0, Q={fit.q_factor:.0f}')
ax.set_xlabel('Frequency (THz)')
ax.set_ylabel('Transmission')
ax.legend()
save_figure(fig, '../assets/ring_lorentzian_fit.png')
plt.show()

## Key takeaway

Gap controls bus–ring coupling and therefore Q. Compare simulated FSR and Q to analytical estimates; large discrepancies usually mean resolution, PML, or geometry issues.